### タスク 1.1: 基礎の設定とベーシックエージェントの作成

Strands Agents フレームワークを使用してカスタマーサポートエージェントのプロトタイプを設定します。このプロトタイプは、エージェントプロトタイプから本番環境対応ソリューションまでの全行程をチェックするための出発点となります。

このタスクを完了すると、エージェントは次の基本アーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab1_strands_ja_jp.png" width="75%"/>
</div>

**画像の説明: ローカルツールでローカルで実行されるシンプルなエージェントプロトタイプ**

依存関係をインストールし、AWS SDK、AgentCore コンポーネント、Strands フレームワークなど必要なすべてのライブラリをインポートして、開発環境を準備します。

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore のインポート
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands のインポート
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent

# ローカルツール
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# セットアップ
boto_session = Session()
REGION = "us-east-1"
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

エージェントを作成する前に、カスタマーサポート機能を強化するローカルツールを調べます。`lab_helpers/lab1_strands_agent.py` を開いて確認し、以下を理解します。

- ツールは `@tool` デコレータを使用してこのファイル内でローカルに定義されます
- 4 つのツール機能とその目的:
  - get_product_info(): 商品情報を取得する
  - get_return_policy(): 特定の商品の返品ポリシーを取得する
  - get_technical_support(): テクニカルサポートガイダンスを提供する
  - web_search(): ウェブで最新情報を検索する
- モックデータの使用方法 (実際のデータベース / API のシミュレーション)
- エージェントの動作を定義するシステムプロンプト

クエリの理解からアクションの実行まで、AI のコア機能を実証するファウンデーショナルカスタマーサポートエージェントを作成します。このエージェントは以下を組み合わせたものです。

- **基盤モデル**: 推論と意思決定を支える「ブレイン」
- **システムプロンプト**: エージェントのパーソナリティとサービス基準を定義する動作指示
- **専門ツール**: 4 つのローカルツール (商品情報、返品ポリシー、テクニカルサポート、ウェブ検索)

エージェントを呼び出すと、エージェントは次のプロセスに従います。
1. **クエリ分析**: エージェントはお客様の質問を分析します
2. **ツールの選択**: エージェントは使用するツールを決定します (存在する場合)
3. **ツールの実行**: エージェントは適切なパラメータを使用して適切なツールを呼び出します
4. **レスポンス合成**: エージェントはツールの結果と知識を組み合わせて役立つレスポンスを作成します
5. **品質チェック**: エージェントはレスポンスがシステムプロンプトの基準を満たしていることを確認します

In [ ]:
# ローカルツールを使用した基本的なエージェントを作成する
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

基本的なエージェントをテストして、お客様からの問い合わせをどのように処理し、ツールをどのように使用するかを確認します。

In [ ]:
# 基本的なエージェントの機能をテストする
print("💬 Testing basic agent...\n")
response = basic_agent("ノートパソコンの返品ポリシーを教えてください。")
print("\n" + "="*50 + "\n")

この初期プロトタイプには、後続のタスクで対処するいくつかの制限があります。

- **永続的メモリなし** - エージェントが以前のセッションの顧客履歴や好みを忘れる
- **ローカルツールのみ** - 共有またはエンタープライズグレードのツール統合なし  
- **アイデンティティ管理なし** - 特定のユーザーに代わって行動できない

### タスク 1.2: メモリでエージェントを強化する

貴重なお客様が最近の注文に関する問題についてサポートチームに問い合わせます。好みを説明し、不満を伝え、エージェントと協力して問題を解決します。3 週間後、関連する質問について再びサポートに問い合わせます。しかし、エージェントは現在の会話セッションのみを記憶して以前のセッションは記憶しないため、好み、履歴、コンテキストなど、すべてを繰り返す必要があります。これにより、以下が作成されます。
- 情報を繰り返さなければならないことで**不満を抱えた顧客**
- 以前のやり取りを基に構築できない**非効率的なサポート**
- 無機質で一般的な応答による**顧客満足度の低下**

Amazon Bedrock AgentCore Memory は、AI エージェントが長期にわたってコンテキストを維持し、重要な事実を記憶し、一貫性のあるパーソナライズされたエクスペリエンスを実現できるようにするマネージドサービスを提供することで、この制限に対処します。AgentCore Memory は次の 2 つのレベルで動作します。
- **短期メモリ**: 即時の会話コンテキストとセッションベースの情報 (Strands Agent フレームワークによって自動的に処理されます)
- **長期メモリ**: 事実、好み、概要など、複数の会話から抽出された永続的な情報 (USER_PREFERENCE および SEMANTIC 戦略を備えた AgentCore Memory サービスを通じて実装されます)

プロトタイプを、以下の例を実行できる顧客対応アシスタントに変換します。
- **「おかえり、サラ」** - リピーターを即座に認識する
- **「先月のノートパソコンに関する問題のフォローアップ」** - 関連する会話をシームレスにつなげる
- **「購入履歴に基づいたおすすめは次のとおりです」** - パーソナライズされた提案を提供する

このタスクを完了すると、エージェントは統合メモリ機能を備えた次のアーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab2_memory_ja_jp.png" width="75%"/>
</div>

**画像の説明: 永続的な顧客コンテキストとパーソナライゼーションのための AgentCore Memory で強化されたエージェント**

**メモリストラテジー設定**: 次の 2 つのインテリジェントな戦略を組み合わせてメモリリソースを作成します。

| 戦略タイプ | 目的 | お客様にとってのメリット |
|---------------|---------|------------------|
| USER_PREFERENCE | 顧客の好みと行動 | 「あなたの好みは...」 |
| SEMANTIC | 事実情報とコンテキスト | 「以前の問題について...」 |

AgentCore Memory は、ActorId を使用して長期メモリメッセージを論理的にグループ化するために名前空間を使用します。
- `support/customer/{actorId}/preferences`: ユーザーの好みのメモリ戦略用
- `support/customer/{actorId}/semantic`: セマンティックメモリ戦略用

In [ ]:
# AgentCore Memory サービス用のメモリクライアントを初期化する
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # SSM パラメータから既存のメモリリソースを取得しようとする
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # 2 つの戦略を持つ新しいメモリリソースを作成する
        strategies = [
            {
                # USER_PREFERENCE 戦略は顧客の好みと行動を取得する
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC 戦略は会話から事実情報を保存する
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # メモリリソースを作成して完了を待つ
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # メモリイベントは 90 日後に期限切れになる
        )
        memory_id = response["id"]
        # 今後の使用のためにメモリ ID を SSM に保存する
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

以前にサポートチームとやり取りをしたことがある「customer_001」という名前のリピーターをシミュレーションします。これは、AgentCore Memory が個々の会話を自動的にリッチで永続的なカスタマーインサイトに変換する方法を示しています。以前のお客様とのやり取りをロードして、AgentCore Memory がそれらを自動的に長期的なカスタマーインサイトに変える様子をご覧ください。

In [ ]:
# 以前の顧客とのやり取りをシードする
previous_interactions = [
    ("MacBook Pro で動画編集中に本体が熱くなる問題が発生しています。", "USER"),
    ("その発熱の問題については対応できます。あなたの MacBook Pro のご注文 (注文番号 #MB-78432) はまだ保証期間内です。", "ASSISTANT"),
    ("ゲーミングヘッドフォンの返品ポリシーを教えてください。競技系 FPS ゲームをやるので低レイテンシーが必要です。", "USER"),
    ("ゲーミングヘッドフォンは 30 日以内であれば返品可能です。競技系 FPS をされているとのことなので、音声レイテンシーの仕様を確認することをおすすめします。", "ASSISTANT"),
    ("プログラミング用に 1200 ドル以下のノートパソコンが必要です。RAM は 16GB 以上が望ましく、Linux との互換性も重視します。ThinkPad が好きです。", "USER"),
    ("了解しました。開発作業には、Linux サポートが優れた ThinkPad E シリーズや Dell XPS モデルをおすすめします。", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

Strands Agents は、厳密に型付けされたイベントコールバックを通じてコンポーネントがエージェントの動作に反応したりエージェントの動作を変更したりできるようにする強力なフックシステムを提供します。これにより、手動で操作しなくてもメモリ操作が自動的に行われます。

お客様がエージェントとやり取りするたびに、次の処理が自動的に行われます。
- 以前のやり取りや好みに基づいて**会話をパーソナライズする**
- **新しいやり取りをメモリに追加**して、今後のパーソナライゼーションを継続的に改善する

フック統合が行うこと:
- **応答前**: 関連する顧客コンテキストと好みを自動的に取得する
- **応答後**: 新しいやり取りを AgentCore Memory に自動的に保存する

メモリフックによる自動顧客コンテキストを有効にします。

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # エージェントが応答する前に実行され、顧客コンテキストを取得するフック
        messages = event.agent.messages
        # ユーザーメッセージのみを処理する (ツールの結果は処理しない)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # USER_PREFERENCE と SEMANTIC の両方について、各戦略の名前空間からメモリを取得する
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # 関連性の高いメモリを上位 3 件取得する
                    )
                    # メモリオブジェクトからテキストコンテンツを抽出する
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # 顧客コンテキストをユーザーメッセージの先頭に追加する
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # エージェントが応答した後に実行され、やり取りをメモリに保存するフック
        try:
            messages = event.agent.messages
            # ユーザーとアシスタントの両方のメッセージがある場合のみ保存する
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # 直近のユーザークエリとアシスタントの応答を見つける
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # やり取りを AgentCore Memory に保存する
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # 両方のフックをエージェントのフックレジストリに登録する
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

メモリ強化エージェントを作成してテストし、顧客コンテキストを取得して応答をパーソナライズする方法を確認します。

In [ ]:
# フックを使用してメモリ強化エージェントを作成する
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# メモリ処理の完了を待つ
print("⏳ Waiting 90 seconds for memory processing to complete...")
time.sleep(90)

# メモリの再現をテストする
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("私のノートパソコンの好みを日本語で教えてください。")
print("\n" + "="*50 + "\n")

### タスク 1.3: ゲートウェイ統合と AgentCore Identity に合わせてスケールする

メモリが用意できたら、強力なツールに焦点を当て、その効影響を拡大します。優れたエージェントには、社内外両方のお客様のために業務を遂行できるように、独自およびサードパーティーの API とデータを最大限に活用できるツールが必要です。しかし、エージェントツールの構築、保護、拡張は難しく、エージェントのプロトタイプから本番環境におけるエージェントの実際のビジネス価値に移行するお客様にとって大きな障害となっています。AgentCore Gateway は、統合**モデルコンテキストプロトコル (MCP)** エンドポイントを通じて AI エージェントが実際のツールを検出、認証、呼び出しできるようにする接続レイヤーとして機能します。これは、何百もの API、リソース、ツールを管理する企業にとって非常に重要です。

主な利点は以下のとおりです。
- インフラストラクチャ管理がない**フルマネージド MCP サーバー**ソリューション
- **既存の API と Lambda 関数の統合**
- 多様なツールでの**統一されたインターフェイス**
- **安全な認証と認可**
- **セマンティックツールの発見**と選択

##### 構築内容:

**ツールの一元化と再利用性:**
- ウェブ検索をローカルツールから集中型の AgentCore Gateway に移行する
- 既存のエンタープライズ Lambda 関数を統合する (保証チェック)
- 複数のエージェントタイプがアクセスできる共有ツールインフラストラクチャを作成する

**エンタープライズグレードのセキュリティ:**
- Cognito 統合を使用した JWT ベース認証を実装する
- ゲートウェイアクセスの安全なインバウンド認可を設定する
- ツールの使用のためのアイデンティティベースのアクセス制御を確立する

これにより、ツールを一元管理して複数のエージェントタイプで再利用できるスケーラブルな基盤が構築され、コードの重複がなくなり、メンテナンスが簡単になります。

AgentCore Identity もこのプロセスに関与しています。これにより、AI エージェントは AWS リソースに安全にアクセスし、Amazon Cognito と連携してインバウンド呼び出し認証に対処する上で役立ちます。また、AI エージェントはアウトバウンド認証を使用してサードパーティーのツールやサービスに安全にアクセスできますが、このラボでは Agentcore Identity のこの機能は使用しません。

<div style="text-align:left">
    <img src="images/architecture_lab3_identity_ja_jp.png" width="75%"/>
</div>

このタスクを完了すると、エージェントは統合ゲートウェイ機能を備えた次のアーキテクチャを使用できるようになります。

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway_ja_jp.png" width="75%"/>
</div>

**画像の説明: 安全で一元的なツール管理とエンタープライズ統合のための、AgentCore Gateway で強化されたエージェント**

AgentCore Gateway を作成し、Lambda 関数を MCP 互換エンドポイントとして公開します。ツールを呼び出す権限のある発信者を検証するには、MCP サーバーの標準である OAuth 認可を使用して**インバウンド認証**を設定します。

### ゲートウェイ認証について理解する

AgentCore Gateway は **OAuth 2.0 と JWT トークン**を使用してツールへのアクセスを保護します。これにより、不正なアプリケーションが Lambda 関数を呼び出すことを防ぎます。

**主要な概念:**

1. **認証プロバイダー**: Amazon Cognito はアイデンティティを管理し、トークンを発行します
2. **クライアント認証情報**: エージェントは client_id と client_secret (アプリケーションのユーザー名 / パスワードなど) を使用します
3. **JWT トークン**: エージェントが承認されていることを証明する短期間のトークン
4. **許可されたクライアント**: Gateway にアクセスできるクライアント ID の許可リスト

**仕組み:**
```
エージェント → Cognito: 「これが私の client_id と client_secret です」
Cognito → エージェント: 「これがあなたの JWT アクセストークンです」
エージェント → ゲートウェイ: 「これが私のトークンです」
ゲートウェイ → Cognito: 「このトークンは有効で、許可されたクライアントからのものですか」
ゲートウェイ → エージェント: 「アクセスが承認されました」
```

**セキュリティ上の注意**: これらの認証情報は事前に作成され、SSM パラメータストアに安全に保存されています。認証情報をコードでハードコーディングすることは決してしないでください

In [ ]:
# SSM パラメータストアから認証設定を取得する
# これらの値は CloudFormation テンプレートによって作成された

# クライアント ID: どのアプリケーションがリクエストを行っているかを識別する
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# 検出 URL: Cognito の OAuth 設定の場所をゲートウェイに伝える
# この URL はトークンエンドポイント、サポートされているスコープなどのメタデータを提供する
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# ゲートウェイの JWT ベース認証を設定する
auth_config = {
    "customJWTAuthorizer": {
        # このクライアント ID からのトークンのみが受け入れられる
        "allowedClients": [machine_client_id],
        # ゲートウェイはこの URL から OAuth メタデータを取得する
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

### AgentCore Gateway を作成する

Gateway は、エージェントとバックエンド Lambda 関数間の安全なプロキシとして機能します。AI エージェント専用に設計された API Gateway として考えます。

**作成しているもの:**
- **ゲートウェイインフラストラクチャ**: コアゲートウェイリソース
- **MCP プロトコル**: ツール通信のための標準プロトコル
- **JWT 認可**: 作成したばかりの認証設定を使用する
- **IAM ロール**: Lambda 関数を呼び出す権限

**次に起こること:**
1. ゲートウェイを作成する (このセル)
2. ツール定義を含む Lambda ターゲットを追加する (次のセル)
   - 1 つの Lambda 関数で複数のツールを処理する: `check_warranty_status` および `web_search`
   - ゲートウェイ はツール名を Lambda に渡し、Lambda は適切なハンドラーにルーティングする
3. エージェントをゲートウェイに接続する

**注**: CloudFormation テンプレートは複数のツール操作を処理できる 1 つの Lambda 関数 (`CustomerSupportLambda`) をデプロイしました。これはツールごとに個別の Lambda 関数をデプロイするよりも効率的です。

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # モデルコンテキストプロトコル
        authorizerType="CUSTOM_JWT",  # 認証に JWT トークンを使用する
        authorizerConfiguration=auth_config,  # 上記の認証設定を使用する
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # ゲートウェイの準備が完了するまで待つ
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # ゲートウェイが既に存在する場合は、それを取得する
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

AgentCore Gateway は呼び出すツールの名前を指定して、Lambda コンテキストを設定します。ツールに渡されるパラメータは、Lambda イベントによって指定されます。これにより、既存のエンタープライズ Lambda 関数 (この場合は `AgentCoreLab-CustomerSupportLambda`) を統合して、複数のエージェントで再利用できます。

API 仕様を使用して Lambda 関数をゲートウェイターゲットとして追加します。

In [ ]:
# Lambda ツールの API 仕様を読み込む
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# ゲートウェイターゲットを作成する
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

Cognito の認証トークンを Strands SDK の MCPClient に統合して、安全な MCP 接続を作成します。

ゲートウェイツールにアクセスするための認証済み MCP クライアントを作成します。

In [ ]:
def get_cognito_client_secret():
    # Cognito API を使用して Cognito クライアントシークレットを取得する
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # クライアント認証情報フローを使用してゲートウェイ認証用の OAuth トークンを取得する
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 クライアント認証情報フロー
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Cognito にアクセストークンをリクエストする
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# OAuth アクセストークン (JWT 形式) を取得する
token_response = get_oauth_token()
access_token = token_response['access_token']

# Bearer トークン認証を使用して MCP クライアントを作成する
mcp_client = MCPClient(
    url=gateway_url,
    headers={"Authorization": f"Bearer {access_token}"},  # Authorization ヘッダーの JWT トークン
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

メモリフック + ローカルツール + ゲートウェイツールすべてを組み合わせます。これにより、一部のツールはローカルのまま (スピードとシンプルさのため)、他のツールはゲートウェイを介して一元化される (再利用性とエンタープライズ統合のため) というハイブリッドアーキテクチャが構築されます。

このアプローチにより、異なるエージェント間でのコードの重複がなくなり、ツール更新の一元管理が可能になります。

In [ ]:
# 顧客コンテキスト用のメモリフックを初期化する
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# ゲートウェイへの MCP クライアント接続を開始する
mcp_client.start()
# ゲートウェイから利用可能なツールを取得する
gateway_tools = mcp_client.list_tools_sync()

# ローカルツールと一元化されたゲートウェイツールを組み合わせる
all_tools = [
    get_product_info,      # ローカルツール
    get_return_policy,     # ローカルツール
    get_technical_support, # ローカルツール
] + gateway_tools          # ゲートウェイツール - web_search, check_warranty_status

# メモリとゲートウェイ統合を備えた強化エージェントを作成する
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # ローカル + ゲートウェイツール
    hooks=[memory_hooks],      # 自動メモリ操作
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

メモリおよびゲートウェイ機能を使用してエージェントをテストします。エージェントがメモリを通じて顧客コンテキストを維持しながらローカルツールと一元化されたゲートウェイツールの両方をシームレスに使用できることを確認します。

テストシナリオには、保証チェック、ウェブ検索、およびメモリ + ゲートウェイ機能の組み合わせ含まれます。

In [ ]:
# ゲートウェイツールをテストする
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("iPhone 15 の最新のトラブルシューティングのヒントを検索してください。")
print("\n" + "="*50 + "\n")

In [ ]:
# 保証チェックをテストする
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("シリアル番号 ABC12345678 の保証状況を確認してください。")
print("\n" + "="*50 + "\n")

In [ ]:
# 組み合わせた機能をテストする
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("またゲーミングヘッドフォンが必要です。最新のレビューも検索してください。")
print("\n" + "="*50 + "\n")

## 次のステップ

🎉 **お疲れ様でした。** ノートブックの演習が完了しました。

以下の作業が完了しました。
- Strands を使用して基本的な AI エージェントプロトタイプを作成した
- AgentCore Memory を使用して拡張し、顧客コンテキストを永続的に把握できるようにした
- 安全で一元的なツール共有のための統合型 AgentCore Gateway
- 本番環境ですぐに使用できるカスタマーサポートシステム全体をテストした

### 次のステップ

1. **このノートブックファイルを閉じます**
2. **ラボの手順に戻ります**
3. **タスク 2 に進みます**。AgentCore ダッシュボードを調べて、リソースを実際に確認します。
